In [55]:
import os
import cv2
import kagglehub
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu

In [50]:
%pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.2 MB/s eta 0:00:00m eta 0:00:010:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 34.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [51]:
from statsmodels.stats.multitest import multipletests

In [2]:
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, remove_small_holes
from scipy.stats import entropy

## Carga de los datos

In [3]:
# Descarga del dataset 
path = kagglehub.dataset_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset")
print("Path to dataset files:", path)

Path to dataset files: /home/xaraxx/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3


In [4]:
base_path = Path(path) / "TB_Chest_Radiography_Database"
normal_path = base_path / "Normal"
tb_path = base_path / "Tuberculosis"

normal_imgs = list(normal_path.glob("*.png"))
tb_imgs = list(tb_path.glob("*.png"))

print(f"Normal: {len(normal_imgs)}")
print(f"Tuberculosis: {len(tb_imgs)}")

Normal: 3500
Tuberculosis: 700


In [5]:
#Building the data frame with the images
data = []

for img_path in normal_imgs:

    img = Image.open(img_path)

    data.append({
        "Nombre_Archivo": img_path.name,
        "path": str(img_path),
        "clase": "Normal",
        "Ancho": img.width,
        "Alto": img.height,
        "Modo": img.mode
    })

for img_path in tb_imgs:

    img = Image.open(img_path)

    data.append({
        "Nombre_Archivo": img_path.name,
        "path": str(img_path),
        "clase": "Tuberculosis",
        "Ancho": img.width,
        "Alto": img.height,
        "Modo": img.mode
    })

df = pd.DataFrame(data)

In [29]:
df


,Nombre_Archivo,path,clase,Ancho,Alto,Modo
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB
...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB


In [19]:
df["path"][4195]

'/home/xaraxx/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Tuberculosis/Tuberculosis-366.png'

In [6]:
def segmentar_imagen(path):
    # Leer la imagen
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"No se pudo leer la imagen: {path}")

    h, w = img.shape
    # Ancho de cada segmento
    ancho_segmento = w // 10

    datos = []

    for i in range(10):
        # Coordenadas del segmento
        x_inicio = i * ancho_segmento
        if i == 9:
            # El último segmento llega hasta el final
            x_fin = w
        else:
            x_fin = (i + 1) * ancho_segmento
        # Extraer segmento
        segmento = img[:, x_inicio:x_fin]
        # Guardar segmento y etiqueta
        datos.append({
            "etiqueta": f"S{i + 1}",
            "segmento": segmento
        })
    # Crear DataFrame
    df = pd.DataFrame(datos)
    return df

def segmentar_imagen(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"No se pudo leer la imagen: {path}")
    h, w = img.shape
    ancho_segmento = w // 10
    datos = []
    for i in range(10):
        x_inicio = i * ancho_segmento
        x_fin = w if i == 9 else (i + 1) * ancho_segmento

        segmento = img[:, x_inicio:x_fin]

        datos.append({
            "imagen": path,
            "etiqueta": f"S{i + 1}",
            "segmento": segmento
        })

    return pd.DataFrame(datos)

In [7]:
df_seg = segmentar_imagen(df["path"][4195])

In [32]:
df.path[4199]


'/home/xaraxx/.cache/kagglehub/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/versions/3/TB_Chest_Radiography_Database/Tuberculosis/Tuberculosis-322.png'

In [34]:
# Segementando todas las imágenes del df
def segmentar_fila(fila):
    # Segmentar la imagen
    df_segmentos = segmentar_imagen(fila["path"])

    # Conservar los metadatos
    df_segmentos["Nombre_Archivo"] = fila["Nombre_Archivo"]
    df_segmentos["clase"] = fila["clase"]

    return df_segmentos

segmentos_lista = df.apply(segmentar_fila, axis=1)

df_segmentos_total = pd.concat(
    segmentos_lista.tolist(),
    ignore_index=True
)

In [35]:
df_segmentos_total

,imagen,etiqueta,segmento,Nombre_Archivo,clase
0,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S1,"[[247, 226, 210, 195, 179, 171, 160, 152, 145,...",Normal-36.png,Normal
1,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S2,"[[98, 98, 97, 97, 99, 98, 97, 97, 99, 97, 96, ...",Normal-36.png,Normal
2,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S3,"[[104, 106, 107, 106, 106, 107, 106, 108, 110,...",Normal-36.png,Normal
3,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S4,"[[117, 118, 118, 117, 119, 119, 118, 118, 117,...",Normal-36.png,Normal
4,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S5,"[[254, 253, 254, 254, 254, 254, 255, 254, 254,...",Normal-36.png,Normal
...,...,...,...,...,...
41995,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S6,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis
41996,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S7,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis
41997,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S8,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis
41998,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S9,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis


In [8]:
def calcular_glcm_segmentos(df):
    """
    Calcula características GLCM para cada segmento de imagen.

    Parámetros
    ----------
    df : pandas.DataFrame
        DataFrame que debe contener una columna 'segmento'
        con las imágenes segmentadas como arrays de NumPy.

    Retorna
    -------
    pandas.DataFrame
        DataFrame original con las características GLCM
        agregadas como nuevas columnas.
    """

    glcm_features_list = []

    for idx, fila in df.iterrows():

        try:
            # 1. Obtener directamente el segmento
            segmento = fila["segmento"]

            # Verificar que exista el segmento
            if segmento is None:
                raise ValueError("El segmento es None")

            # 2. Asegurar que sea uint8
            segmento_uint8 = segmento.astype(np.uint8)

            # 3. Calcular la matriz GLCM
            glcm = graycomatrix(
                segmento_uint8,
                distances=[1],
                angles=[
                    0,
                    np.pi / 4,
                    np.pi / 2,
                    3 * np.pi / 4
                ],
                levels=256,
                symmetric=True,
                normed=True
            )
            # Entropy of Shannon
            hist = cv2.calcHist(segmento, [0], None, [256], [0, 256]).flatten()
            hist_prob = hist/hist.sum()
            segment_entropy = entropy(hist_prob, base=2)

            # 4. Calcular características GLCM
            features = {
                "glcm_contrast": graycoprops(
                    glcm, "contrast"
                ).mean(),

                "glcm_dissimilarity": graycoprops(
                    glcm, "dissimilarity"
                ).mean(),

                "glcm_homogeneity": graycoprops(
                    glcm, "homogeneity"
                ).mean(),

                "glcm_energy": graycoprops(
                    glcm, "energy"
                ).mean(),

                "glcm_correlation": graycoprops(
                    glcm, "correlation"
                ).mean(),
                "entropy":segment_entropy
            }
            glcm_features_list.append(features)
            

        except Exception as e:

            print(
                f"Error inesperado en el segmento "
                f"de la fila {idx}: {e}"
            )

            # Si ocurre un error, guardar NaN
            glcm_features_list.append({
                "glcm_contrast": np.nan,
                "glcm_dissimilarity": np.nan,
                "glcm_homogeneity": np.nan,
                "glcm_energy": np.nan,
                "glcm_correlation": np.nan
            })

    # 5. Crear DataFrame con las características GLCM
    df_features = pd.DataFrame(glcm_features_list)

    # 6. Concatenar con el DataFrame original
    df_resultado = pd.concat(
        [
            df.reset_index(drop=True),
            df_features
        ],
        axis=1
    )

    return df_resultado

In [9]:
# Cálculo para una sola imagen
df_feat_seg = calcular_glcm_segmentos(df_seg)

In [36]:
# Cálculo de características para todo el dataframe segmentado
df_glcm = calcular_glcm_segmentos(df_segmentos_total)

In [37]:
df_glcm

,imagen,etiqueta,segmento,Nombre_Archivo,clase,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation,entropy
0,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S1,"[[247, 226, 210, 195, 179, 171, 160, 152, 145,...",Normal-36.png,Normal,12.339654,1.454156,0.696548,0.402516,0.995402,3.811447
1,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S2,"[[98, 98, 97, 97, 99, 98, 97, 97, 99, 97, 96, ...",Normal-36.png,Normal,9.466386,1.680456,0.549342,0.137583,0.997663,3.069132
2,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S3,"[[104, 106, 107, 106, 106, 107, 106, 108, 110,...",Normal-36.png,Normal,32.128936,3.565082,0.301432,0.024472,0.992893,3.468433
3,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S4,"[[117, 118, 118, 117, 119, 119, 118, 118, 117,...",Normal-36.png,Normal,25.538499,3.265169,0.301818,0.023452,0.996673,3.914607
4,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S5,"[[254, 253, 254, 254, 254, 254, 255, 254, 254,...",Normal-36.png,Normal,27.107420,3.459329,0.287782,0.025371,0.996406,1.063432
...,...,...,...,...,...,...,...,...,...,...,...
41995,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S6,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis,44.862882,3.912039,0.278600,0.022602,0.993770,0.000000
41996,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S7,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis,33.410750,3.699501,0.284725,0.023152,0.993476,0.000000
41997,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S8,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis,71.262358,3.800140,0.350717,0.094707,0.992798,0.000000
41998,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,S9,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Tuberculosis-322.png,Tuberculosis,110.563182,1.322675,0.893329,0.836988,0.972794,0.000000


In [39]:
caracteristicas = [
    "glcm_contrast",
    "glcm_dissimilarity",
    "glcm_homogeneity",
    "glcm_energy",
    "glcm_correlation",
    "entropy"
]

# Verificar que existan todas las columnas
faltantes = [
    c for c in caracteristicas
    if c not in df_glcm.columns
]

if faltantes:
    raise ValueError(
        f"Faltan estas características en df_glcm: {faltantes}"
    )

print(df_glcm[["Nombre_Archivo", "clase", "etiqueta"] + caracteristicas].head())

  Nombre_Archivo   clase etiqueta  glcm_contrast  glcm_dissimilarity  \
0  Normal-36.png  Normal       S1      12.339654            1.454156   
1  Normal-36.png  Normal       S2       9.466386            1.680456   
2  Normal-36.png  Normal       S3      32.128936            3.565082   
3  Normal-36.png  Normal       S4      25.538499            3.265169   
4  Normal-36.png  Normal       S5      27.107420            3.459329   

   glcm_homogeneity  glcm_energy  glcm_correlation   entropy  
0          0.696548     0.402516          0.995402  3.811447  
1          0.549342     0.137583          0.997663  3.069132  
2          0.301432     0.024472          0.992893  3.468433  
3          0.301818     0.023452          0.996673  3.914607  
4          0.287782     0.025371          0.996406  1.063432  


In [42]:
caracteristicas = [
    "glcm_contrast",
    "glcm_dissimilarity",
    "glcm_homogeneity",
    "glcm_energy",
    "glcm_correlation",
    "entropy"
]

# Trabajar directamente con los segmentos, sin agrupar por imagen
df_segmentos = df_glcm.copy()

# Verificar las clases disponibles
print(df_segmentos["clase"].value_counts())

# Comprobar las columnas necesarias
faltantes = [
    c for c in caracteristicas + ["clase", "etiqueta"]
    if c not in df_segmentos.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas: {faltantes}")

# Conservar únicamente las dos clases que vamos a comparar
df_segmentos = df_segmentos[
    df_segmentos["clase"].isin(["Normal", "Tuberculosis"])
].copy()

clase
Normal          35000
Tuberculosis     7000
Name: count, dtype: int64


In [52]:
def analizar_segmentos(df, caracteristicas):
    resultados = []

    for caracteristica in caracteristicas:

        # Extraer las características de cada clase
        normal = pd.to_numeric(
            df.loc[df["clase"] == "Normal", caracteristica],
            errors="coerce"
        ).to_numpy()

        tuberculosis = pd.to_numeric(
            df.loc[df["clase"] == "Tuberculosis", caracteristica],
            errors="coerce"
        ).to_numpy()

        # Eliminar valores NaN e infinitos
        normal = normal[np.isfinite(normal)]
        tuberculosis = tuberculosis[np.isfinite(tuberculosis)]

        n1 = len(normal)
        n2 = len(tuberculosis)

        if n1 < 2 or n2 < 2:
            print(f"Se omite {caracteristica}: datos insuficientes.")
            continue

        # Estadísticas descriptivas
        media_normal = np.mean(normal)
        media_tb = np.mean(tuberculosis)

        sd_normal = np.std(normal, ddof=1)
        sd_tb = np.std(tuberculosis, ddof=1)

        diferencia = media_normal - media_tb

        # 1. ANOVA de un factor
        f_stat, p_anova = stats.f_oneway(
            normal,
            tuberculosis
        )

        # 2. Prueba t de Student: varianzas iguales
        t_stat, p_tstudent = stats.ttest_ind(
            normal,
            tuberculosis,
            equal_var=True
        )

        # 3. Prueba t de Welch: varianzas diferentes
        t_welch, p_welch = stats.ttest_ind(
            normal,
            tuberculosis,
            equal_var=False
        )

        # 4. d de Cohen
        sd_pooled = np.sqrt(
            (
                (n1 - 1) * sd_normal**2
                + (n2 - 1) * sd_tb**2
            ) / (n1 + n2 - 2)
        )

        cohen_d = (
            diferencia / sd_pooled
            if sd_pooled > 0
            else np.nan
        )

        # 5. U de Mann-Whitney
        u_stat, p_mw = stats.mannwhitneyu(
            normal,
            tuberculosis,
            alternative="two-sided"
        )

        # 6. Correlación biserial por rangos
        # Positiva: Normal tiende a tener valores mayores.
        # Negativa: Tuberculosis tiende a tener valores mayores.
        rank_biserial = (2 * u_stat / (n1 * n2)) - 1

        resultados.append({
            "caracteristica": caracteristica,
            "n_normal": n1,
            "n_tuberculosis": n2,
            "media_normal": media_normal,
            "media_tuberculosis": media_tb,
            "sd_normal": sd_normal,
            "sd_tuberculosis": sd_tb,
            "diferencia_medias": diferencia,
            "F_anova": f_stat,
            "p_anova": p_anova,
            "t_student": t_stat,
            "p_t_student": p_tstudent,
            "t_welch": t_welch,
            "p_welch": p_welch,
            "cohen_d": cohen_d,
            "U_mannwhitney": u_stat,
            "p_mannwhitney": p_mw,
            "rank_biserial": rank_biserial
        })

    df_resultados = pd.DataFrame(resultados)

    if df_resultados.empty:
        return df_resultados

    # Corrección por comparaciones múltiples:
    # Benjamini-Hochberg, aplicada por separado a cada prueba.
    for columna in [
        "p_anova",
        "p_t_student",
        "p_welch",
        "p_mannwhitney"
    ]:
        df_resultados[f"{columna}_ajustado"] = multipletests(
            df_resultados[columna],
            method="fdr_bh"
        )[1]

    return df_resultados

In [53]:
df_resultados = analizar_segmentos(
    df_segmentos,
    caracteristicas
)

#print(df_resultados.round(5).to_string(index=False))

In [54]:
df_resultados

,caracteristica,n_normal,n_tuberculosis,media_normal,media_tuberculosis,sd_normal,sd_tuberculosis,diferencia_medias,F_anova,p_anova,...,t_welch,p_welch,cohen_d,U_mannwhitney,p_mannwhitney,rank_biserial,p_anova_ajustado,p_t_student_ajustado,p_welch_ajustado,p_mannwhitney_ajustado
0,glcm_contrast,35000,7000,41.717180,27.534156,49.641668,34.635012,14.183023,520.704203,1.483206e-114,...,28.844353,1.399417e-177,0.298770,162257789.0,0.000000e+00,0.324553,1.779848e-114,1.779848e-114,2.099126e-177,0.000000e+00
1,glcm_dissimilarity,35000,7000,2.849749,2.117791,0.727733,0.877095,0.731957,5487.357491,0.000000e+00,...,65.460209,0.000000e+00,0.969892,188355626.0,0.000000e+00,0.537597,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
2,glcm_homogeneity,35000,7000,0.403816,0.529153,0.088655,0.113703,-0.125337,10527.730191,0.000000e+00,...,-87.083933,0.000000e+00,-1.343411,40777755.0,0.000000e+00,-0.667120,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
3,glcm_energy,35000,7000,0.065688,0.099493,0.080536,0.123023,-0.033804,840.874447,4.504855e-183,...,-22.063607,7.368023e-105,-0.379671,67395546.0,0.000000e+00,-0.449832,6.757282e-183,6.757282e-183,8.841627e-105,0.000000e+00
4,glcm_correlation,35000,7000,0.991745,0.991666,0.013621,0.010626,0.000079,0.211686,6.454513e-01,...,0.541909,5.878912e-01,0.006024,130125776.0,1.796094e-16,0.062251,6.454513e-01,6.454513e-01,5.878912e-01,1.796094e-16
5,entropy,35000,7000,2.934584,2.362608,1.728794,1.725782,0.571976,638.908386,0.000000e+00,...,25.306030,0.000000e+00,0.330949,145812480.0,0.000000e+00,0.190306,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00


In [57]:
# Características calculadas para cada segmento
metricas = [
    "glcm_contrast",
    "glcm_dissimilarity",
    "glcm_homogeneity",
    "glcm_energy",
    "glcm_correlation",
    "entropy"
]

# Segmentos de las imágenes
regiones = [f"S{i}" for i in range(1, 11)]

# Preparar el DataFrame
df = df_segmentos.copy()

# Comprobar que existen las columnas requeridas
columnas_requeridas = ["clase", "etiqueta"] + metricas
faltantes = [c for c in columnas_requeridas if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas en df_segmentos: {faltantes}")

# Calcular la prueba de Mann-Whitney
resultados = []

for region in regiones:

    subset = df[df["etiqueta"] == region]

    grupo_normal = subset.loc[
        subset["clase"] == "Normal"
    ]

    grupo_tb = subset.loc[
        subset["clase"] == "Tuberculosis"
    ]

    for metrica in metricas:

        # Convertir a valores numéricos y eliminar NaN e infinitos
        x = pd.to_numeric(
            grupo_normal[metrica], errors="coerce"
        ).to_numpy(dtype=float)

        y = pd.to_numeric(
            grupo_tb[metrica], errors="coerce"
        ).to_numpy(dtype=float)

        x = x[np.isfinite(x)]
        y = y[np.isfinite(y)]

        # Evitar errores cuando no hay suficientes observaciones
        if len(x) == 0 or len(y) == 0:
            continue

        # Prueba U de Mann-Whitney
        result = mannwhitneyu(
            x,
            y,
            alternative="two-sided"
        )

        resultados.append({
            "region": region,
            "metrica": metrica,
            "U_stat": result.statistic,
            "p_value": result.pvalue,
            "n_normal": len(x),
            "n_tb": len(y)
        })

# Crear DataFrame de resultados
df_mannwhitney = pd.DataFrame(resultados)

# Calcular correlación biserial de rangos
df_mannwhitney["rank_biserial"] = (
    2 * df_mannwhitney["U_stat"]
    / (
        df_mannwhitney["n_normal"]
        * df_mannwhitney["n_tb"]
    )
    - 1
)

# Ordenar por valor p, de menor a mayor
df_mannwhitney = (
    df_mannwhitney
    .sort_values("p_value")
    .reset_index(drop=True)
)


In [58]:
df_mannwhitney

,region,metrica,U_stat,p_value,n_normal,n_tb,rank_biserial
0,S6,glcm_energy,136032.0,1.277711e-302,3500,700,-0.888953
1,S6,glcm_homogeneity,155951.0,9.935661e-292,3500,700,-0.872693
2,S5,glcm_energy,182023.0,8.887582e-278,3500,700,-0.851410
3,S5,glcm_homogeneity,196988.0,6.330850e-270,3500,700,-0.839193
4,S7,glcm_energy,283273.0,7.385817e-227,3500,700,-0.768757
5,S7,glcm_homogeneity,285597.0,9.468377e-226,3500,700,-0.766860
6,S4,glcm_homogeneity,294699.0,1.945352e-221,3500,700,-0.759429
7,S8,glcm_homogeneity,303119.0,1.742825e-217,3500,700,-0.752556
8,S6,glcm_dissimilarity,2135058.0,5.374281e-212,3500,700,0.742904
9,S3,glcm_homogeneity,316307.0,2.288209e-211,3500,700,-0.741790


In [40]:
df_imagenes = (
    df_glcm
    .groupby(["Nombre_Archivo", "clase"], as_index=False)[caracteristicas]
    .mean()
)

print("Número de imágenes:", len(df_imagenes))
print(df_imagenes["clase"].value_counts())
print(df_imagenes.head())

Número de imágenes: 4200
clase
Normal          3500
Tuberculosis     700
Name: count, dtype: int64
    Nombre_Archivo   clase  glcm_contrast  glcm_dissimilarity  \
0     Normal-1.png  Normal      27.713443            2.119133   
1    Normal-10.png  Normal      31.140601            3.163427   
2   Normal-100.png  Normal      24.322457            1.891630   
3  Normal-1000.png  Normal      35.787073            3.117716   
4  Normal-1001.png  Normal      24.149381            2.908329   

   glcm_homogeneity  glcm_energy  glcm_correlation   entropy  
0          0.479374     0.073915          0.993321  2.321240  
1          0.359169     0.046431          0.995156  3.186316  
2          0.491468     0.060000          0.991839  2.186937  
3          0.373143     0.066002          0.993489  3.161284  
4          0.379861     0.087227          0.996566  2.216945  


In [41]:
df_imagenes

,Nombre_Archivo,clase,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation,entropy
0,Normal-1.png,Normal,27.713443,2.119133,0.479374,0.073915,0.993321,2.321240
1,Normal-10.png,Normal,31.140601,3.163427,0.359169,0.046431,0.995156,3.186316
2,Normal-100.png,Normal,24.322457,1.891630,0.491468,0.060000,0.991839,2.186937
3,Normal-1000.png,Normal,35.787073,3.117716,0.373143,0.066002,0.993489,3.161284
4,Normal-1001.png,Normal,24.149381,2.908329,0.379861,0.087227,0.996566,2.216945
...,...,...,...,...,...,...,...,...
4195,Tuberculosis-95.png,Tuberculosis,59.134778,2.894386,0.486974,0.073205,0.984346,0.343756
4196,Tuberculosis-96.png,Tuberculosis,60.191568,2.867651,0.486417,0.070223,0.981611,3.030965
4197,Tuberculosis-97.png,Tuberculosis,21.062523,1.979045,0.521469,0.069266,0.988721,2.255025
4198,Tuberculosis-98.png,Tuberculosis,18.659407,1.872534,0.532615,0.073077,0.989145,2.107485


In [10]:
def calcular_glcm_dataset(df):
    """
    Calcula las características GLCM promedio para cada imagen 
    """
    glcm_features_list = []
    
    for idx, fila in df.iterrows():
        try:
            # 1. Leer la imagen directamente en escala de grises (maneja RGB y L)
            ruta_img = str(fila['path'])
            img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                print(f"Error: No se pudo cargar la imagen en la fila {idx}: {ruta_img}")
                # Insertar NaNs si la imagen está corrupta o no existe
                glcm_features_list.append({
                    "glcm_contrast": np.nan, "glcm_dissimilarity": np.nan,
                    "glcm_homogeneity": np.nan, "glcm_energy": np.nan, "glcm_correlation": np.nan
                })
                continue
            
            # Asegurar formato uint8 necesario para graycomatrix
            img_uint8 = img.astype(np.uint8)
            
            # 2. Calcular la matriz GLCM (distancia 1, 4 ángulos)
            glcm = graycomatrix(img_uint8, distances=[1],
                                angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                                levels=256, symmetric=True, normed=True)
            
            # 3. Extraer las propiedades promediando los 4 ángulos
            features = {
                "glcm_contrast": graycoprops(glcm, "contrast").mean(),
                "glcm_dissimilarity": graycoprops(glcm, "dissimilarity").mean(),
                "glcm_homogeneity": graycoprops(glcm, "homogeneity").mean(),
                "glcm_energy": graycoprops(glcm, "energy").mean(),
                "glcm_correlation": graycoprops(glcm, "correlation").mean()
            }
            
            glcm_features_list.append(features)
            
        except Exception as e:
            print(f"Error inesperado en la fila {idx}: {e}")
            glcm_features_list.append({
                "glcm_contrast": np.nan, "glcm_dissimilarity": np.nan,
                "glcm_homogeneity": np.nan, "glcm_energy": np.nan, "glcm_correlation": np.nan
            })
            
    # 4. Convertir la lista de características en un DataFrame
    df_features = pd.DataFrame(glcm_features_list)
    
    # 5. Concatenar horizontalmente con tu DataFrame original
    df_resultado = pd.concat([df.reset_index(drop=True), df_features], axis=1)
    
    return df_resultado

In [11]:
def calcular_caracteristicas_forma_dataset(df):
    """
    Segmenta la región más densa de cada imagen en la columna 'path'
    y extrae sus características morfológicas de forma segura.
    """
    shape_features_list = []
    
    for idx, fila in df.iterrows():
        try:
            # 1. Leer imagen en escala de grises
            ruta_img = str(fila['path'])
            img = cv2.imread(ruta_img, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                print(f"[-] Fila {idx}: No se pudo cargar la imagen en la ruta: {ruta_img}")
                raise ValueError("Ruta inválida o archivo corrupto")
                
            img_uint8 = img.astype(np.uint8)
            
            # 2. Segmentación con Otsu (con respaldo si se queda vacía)
            thresh = threshold_otsu(img_uint8)
            binary = img_uint8 > (thresh * 1.05)
            
            # Si la optimización del 5% borra toda la segmentación, usamos Otsu normal
            if not np.any(binary):
                binary = img_uint8 > thresh
                
            # Asegurar tipo bool para evitar conflictos en scikit-image
            binary = binary.astype(bool)
            binary = remove_small_objects(binary, min_size=80)
            binary = remove_small_holes(binary, area_threshold=200)
            
            labeled = label(binary)
            regions = regionprops(labeled, intensity_image=img_uint8)
            
            # Si aún así no hay regiones detectadas
            if not regions:
                shape_features_list.append({
                    "shape_area": np.nan, "shape_perimeter": np.nan, "shape_circularity": np.nan,
                    "shape_extent": np.nan, "shape_solidity": np.nan, 
                    "shape_centroid_x": np.nan, "shape_centroid_y": np.nan
                })
                continue
            
            # 3. Seleccionar la región más representativa
            candidates = [r for r in regions if 80 < r.area < img_uint8.size * 0.08]
            if candidates:
                lesion = max(candidates, key=lambda r: r.mean_intensity * r.solidity)
            else:
                lesion = max(regions, key=lambda r: r.area)
            
            # 4. Calcular métricas de forma
            perimeter = lesion.perimeter
            circ = (4 * np.pi * lesion.area / (perimeter**2)) if perimeter > 0 else np.nan
            
            features = {
                "shape_area": float(lesion.area),
                "shape_perimeter": float(perimeter),
                "shape_circularity": float(circ),
                "shape_extent": float(lesion.extent),
                "shape_solidity": float(lesion.solidity),
                "shape_centroid_x": float(lesion.centroid[1]), # Columna
                "shape_centroid_y": float(lesion.centroid[0])  # Fila
            }
            
            shape_features_list.append(features)
            
        except Exception as e:
            # Imprime el error exacto para saber qué falló en el procesamiento interno
            print(f"[!] Error en la fila {idx}: {e}")
            shape_features_list.append({
                "shape_area": np.nan, "shape_perimeter": np.nan, "shape_circularity": np.nan,
                "shape_extent": np.nan, "shape_solidity": np.nan, 
                "shape_centroid_x": np.nan, "shape_centroid_y": np.nan
            })
            
    # 5. Convertir a DataFrame y concatenar
    df_features = pd.DataFrame(shape_features_list)
    df_resultado = pd.concat([df.reset_index(drop=True), df_features], axis=1)
    
    return df_resultado

In [10]:
imgs_features_glcm_df = calcular_glcm_dataset(df)

In [ ]:
imgs_features_glcm_plus_shape_base = calcular_caracteristicas_forma_dataset(df)

/tmp/ipykernel_183474/2498333901.py:30: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  binary = remove_small_objects(binary, min_size=80)
/tmp/ipykernel_183474/2498333901.py:31: FutureWarning: Parameter `area_threshold` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_holes`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  binary = remove_small_holes(binary, area_threshold=200)
/tmp/ipykernel_183474/249833390

In [24]:
imgs_features_glcm_plus_shape_base

,Nombre_Archivo,path,clase,Ancho,Alto,Modo,shape_area,shape_perimeter,shape_circularity,shape_extent,shape_solidity,shape_centroid_x,shape_centroid_y
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,1025.0,402.988853,0.079314,0.187112,0.362191,4.434146,168.102439
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,96.0,54.041631,0.413071,0.410256,0.662069,468.083333,49.385417
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,145289.0,5833.010640,0.053661,0.554234,0.587824,242.787114,287.845783
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,81.0,76.621320,0.173379,0.397059,0.632812,19.530864,0.444444
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,131047.0,5290.508114,0.058836,0.499905,0.550856,263.485078,290.371020
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,4625.0,315.656854,0.583299,0.997627,0.998704,18.507459,450.641946
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,186865.0,3784.237770,0.163976,0.712833,0.781221,306.204099,276.933679
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L,128.0,66.769553,0.360797,0.338624,0.711111,316.312500,222.656250
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,20201.0,1007.641702,0.250018,0.368793,0.597928,62.625662,97.385278


In [23]:
imgs_features_glcm_df

,Nombre_Archivo,path,clase,Ancho,Alto,Modo,glcm_contrast,glcm_dissimilarity,glcm_homogeneity,glcm_energy,glcm_correlation
0,Normal-36.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,45.755570,3.082551,0.383555,0.067999,0.994814
1,Normal-332.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,10.787725,1.691166,0.505183,0.044947,0.995822
2,Normal-840.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,29.687370,3.071503,0.350901,0.046374,0.996419
3,Normal-1965.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,32.477240,3.048858,0.356249,0.042570,0.996241
4,Normal-1304.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Normal,512,512,RGB,28.668172,3.198054,0.345249,0.034219,0.997167
...,...,...,...,...,...,...,...,...,...,...,...
4195,Tuberculosis-366.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,44.758398,2.735331,0.546848,0.072755,0.994984
4196,Tuberculosis-555.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,21.559046,1.459540,0.745914,0.511174,0.998140
4197,Tuberculosis-667.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,L,54.238006,2.721444,0.528639,0.167146,0.992908
4198,Tuberculosis-217.png,/home/xaraxx/.cache/kagglehub/datasets/tawsifu...,Tuberculosis,512,512,RGB,16.668138,2.044816,0.530283,0.056111,0.998379


In [27]:
imgs_features_glcm_df.to_csv("dataset_caracteristicas_glcm.csv", index=False)

In [ ]:
imgs_features_glcm_plus_shape_base.to_csv("", index=False)